# 2. 에이전트 RAG


- 에이전트 RAG는 RAG에 LLM의 의사결정 능력을 결합한 시스템입니다.
- ReAct 방법론은 대표적인 에이전트 구현 방식 중 하나입니다.
- 2개의 서로 다른 PDF파일로부터 2개의 검색기를 만들어 ReAct 에이전트와 연결하여 복잡한 질문을 처리할 수 있는 에이전트 RAG를 구현해보겠습니다.


In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.tools.retriever import create_retriever_tool
from langchain import hub
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

- PyMuPDFLoader: PDF 문서를 읽어들이고 텍스트를 추출하는 도구입니다.
- RecursiveCharacterTextSplitter: 긴 문서를 의미 있는 단위로 분할하는 도구로, 문장과 단락의 문맥을 보존하며 텍스트를 청크 단위로 나눕니다.
- OllamaEmbeddings: 임베딩 모델을 사용해 텍스트를 벡터로 변환합니다.
- Chroma: 벡터화된 텍스트를 저장하고 검색하기 위한 벡터 데이터베이스입니다.
- create_retriever_tool: 벡터 검색을 ReAct에이전트의 도구로 변환합니다.
- hub: LangChain의 프롬프트 템플릿 저장소에 접근합니다.
- ChatOpenAI: 오픈AI의 챗GPT 모델을 활용하기 위한 인터페이스입니다.
- AgentExecutor, create_react_agent: ReAct 에이전트를 생성하고 실행하는 핵심 컴포넌트입니다.
- PromptTemplate: 에이전트의 프롬프트를 템플릿화하여 관리합니다.


#


# 에이전트 도구 만들기


In [2]:
embd = OllamaEmbeddings(model="bge-m3")

- PDF 문서를 벡터 데이터베이스로 변환하고 사용자 질의로부터 유사한 문서를 반환하는 검색기 객체인 retriever를 생성하는 함수 create_pdf_retriever를 구현합니다.


In [3]:
def create_pdf_retriever(
    pdf_path: str,  # PDF 파일 경로
    persist_directory: str,  # 벡터 스토어 저장 경로
    embedding_model: OllamaEmbeddings,  # 임베딩 모델
    chunk_size: int = 512,  # 텍스트 청크 크기
    chunk_overlap: int = 0,  # 청크 오버랩 크기
) -> Chroma.as_retriever:
    # PDF 파일 로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # 청킹
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model,
    )

    return vectorstore.as_retriever()

- 이 함수를 이용하여 일본 ICT 정책에 대해 검색하는 검색기와 미국 ICT 정책에 대해 검색하는 검색기를 각각 만들어 봅시다.


In [4]:
# 일본 ICT 정책 데이터베이스 생성
retriever_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    persist_directory="db_ict_policy_japan_2024",
    embedding_model=embd,
)

# 미국 ICT 정책 데이터베이스 생성
retriever_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    persist_directory="db_ict_policy_usa_2024",
    embedding_model=embd,
)

- 같은 경로를 사용한다면 두번째 문서를 처리할 때 첫번째 문서의 데이터가 덮어써지거나 섞일 수 있기 때문에 경로를 분리
- create_retriever_tool 함수를 하용해 ReAct 에이전트가 사용할 수 있는 검색 도구로 변환


In [5]:
jp_engine = create_retriever_tool(
    retriever=retriever_japan,
    name="japan_ict",
    description="일본의 ICT 시장 동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)
usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name="usa_ict",
    description="미국의 ICT 시장 동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

tools = [jp_engine, usa_engine]

- create_retriever_tool은 retriever 객체를 ReAct 에이전트가 활용할 수 있는 도구형태로 변환해주는 함수입니다.
- description에는 각 검색기의 상세한 용도를 작성해야 합니다.


# 에이전트 프롬프트 설정

- 랭체인에서 ReAct 에이전트를 동작시킬 때 기본값으로 제공하는 프롬프트가 있습니다.
- 너무 단순합니다. 제대로 된 동작을 유도하려면 사용자가 조금 더 자세하게 작성하는 것이 좋습니다.


In [6]:
# 랭체인 기본 제공 프롬프트
prompt_react = hub.pull("hwchase17/react")
print(prompt_react)
print("--프롬프트 끝--")

c:\workspace\python\rag_master\.venv\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'
--프롬프트 끝--


주어진 질문들에 대해 최선을 다해 답변하세요. 다음과 같은 도구들을 사용할 수 있습니다:  
{tools}  
다음 형식을 사용하세요:  
Question: 답변해야 할 입력 질문  
Thought: 무엇을 해야 할지 항상 고민해야 합니다.  
Action: 수행할 행동(반드시 [{tool_names}] 중 하나여야 함)  
Action Input: 행동에 필요한 입력값  
Observation: 행동의 결과  
...(이 Thought/Action/Action Input/Observation 과정은 N번 반복될 수 있습니다.)  
Thought: 이제 최종 답을 알았습니다.  
Final Answer: 원래 입력 질문에 대한 최종 답변  
시작!  
Quesion: {input}  
Thought: {agent_scratchpad}


- 프롬프트 사이에 중괄호 {}로 감싼 부분은 변수에 해당합니다.
- 실제 실행 시에는 각 변수에 적절한 값이 채워지는 구조입니다.


- {tools}:
  - 에이전트가 사용할 수 있는 도구들의 설명이 포함된 목록입니다.
  - create_retriever_tools()에서 정의한 도구들의 이름과 설명이 해당위치에 들어갑니다.
- {tool_names}:
  - 에이전트가 선택할 수 있는 도구의 이름들을 기재합니다.
  - 여기에는 설명없이 도구들의 이름만 리스트 형태로 들어갑니다.
- {input}:
  - 사용자가 현재 물어본 질문이 이 부분에 들어갑니다.
- {agent_scratchpad}:
  - 에이전트의 모든 사이클(Throught/Action/Observation의 기록)이 이 부분에 누적됩니다.
  - 이를 통해 에이전트는 이전의 사이클들을 참고하여 다음 행동을 결정할 수 있습니다.


- 먼저 에이전트가 사용할 수 있는 도구들을 설명하고, 문제 해결을 위해 "Throught/Action/Action Input/Observation"의 사이클을 N번 반복할 수 있다고 안내합니다.
- 주목할 점은 Throught 단계를 각 Action 전후에 하도록 지시한다는 점입니다.
- 이는 에이전트가 행동을 취하기 전에 충분히 고민하고, 또 행동의 결과를 관찰한 후에도 다시 한번 생각하면서 사이클을 돌도록 유도합니다.
- 사용자의 Question에 답하기 위한 Observation이 충분히 취합되면, Final Answer전에 이제 최종 답을 알았습니다. 라는 명시적인 마지막 생각 단계를 작성 하도록 하여 에이전트가 자신의 결론에 확신을 갖는 경우에 답변하도록 설계되었습니다.


In [7]:
template = """다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요.
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Throught/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Throught: 이제 최종 답변을 알겠습니다.
Final Answer: 원래 입력된 질문에 대한 최종 답변

## 추가적인 주의사항
- 반드시 [Throught -> Action -> Action Input format] 이 사이클의 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것이 중요합니다.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

- 기본으로 제공되는 프롬프트와 달리 ## 추가적인 주의사항이라는 내용이 추가되었습니다.
- 더 나은 성능을 얻기위해 추가한 내용입니다.
